In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.sparse import save_npz
from sklearn.feature_extraction.text import TfidfVectorizer

DATA_FOLDER = Path("../../data")
INPUT_PATH = DATA_FOLDER / "processed" / "processed_data.csv"
OUTPUT_FOLDER = DATA_FOLDER / "features" / "tfidf"

MATRIX_PATH = OUTPUT_FOLDER / "tfidf_matrix.npz"
VECTORIZER_PATH = OUTPUT_FOLDER / "tfidf_vectorizer.joblib"
DOCUMENTS_PATH = OUTPUT_FOLDER / "tfidf_documents.csv"
VOCABULARY_PATH = OUTPUT_FOLDER / "tfidf_vocabulary.csv"

print(f"Input: {INPUT_PATH.resolve()}")
print(f"Output folder: {OUTPUT_FOLDER.resolve()}")

Input: C:\Works\uts-ir\src\data\processed\processed_data.csv
Output folder: C:\Works\uts-ir\src\data\features\tfidf


In [2]:
# Load Dataset

df = pd.read_csv(INPUT_PATH)

df.head()

,id,title,abstract,title_translated,abstract_translated,processed_token_title,processed_token_abstract,processed_title,processed_abstract
0,2541-3279-1,hubungan antara promosi perpustakaan melalui m...,penelitian ini dilatarbelakangi oleh perkemban...,hubungan antara promosi perpustakaan melalui m...,penelitian ini dilatarbelakangi oleh perkemban...,"['hubung', 'promosi', 'pustaka', 'lalu', 'maja...","['teliti', 'dilatarbelakangi', 'kembang', 'jam...",hubung promosi pustaka lalu majalah dinding mi...,teliti dilatarbelakangi kembang jaman makin ma...
1,2541-3279-2,hubungan antara penggunaan search features e-j...,penelitian ini membahas hubungan antara penggu...,hubungan antara penggunaan search features e-j...,penelitian ini membahas hubungan antara penggu...,"['hubung', 'guna', 'search', 'features', 'jour...","['teliti', 'bahas', 'hubung', 'guna', 'search'...",hubung guna search features journals penuh but...,teliti bahas hubung guna search features journ...
2,2541-3279-3,hubungan antara ketersediaan fasilitas perpust...,penelitian ini membahas hubungan antara keters...,hubungan antara ketersediaan fasilitas perpust...,penelitian ini membahas hubungan antara keters...,"['hubung', 'sedia', 'fasilitas', 'pustaka', 'm...","['teliti', 'bahas', 'hubung', 'sedia', 'fasili...",hubung sedia fasilitas pustaka minat kunjung s...,teliti bahas hubung sedia fasilitas pustaka mi...
3,2541-3279-4,pengaruh kinerja terhadap layanan prima di cen...,masalah yang menjadi kajian dalam penelitian a...,pengaruh kinerja terhadap layanan prima di cen...,masalah yang menjadi kajian dalam penelitian a...,"['pengaruh', 'kerja', 'layan', 'prima', 'cente...","['masalah', 'jadi', 'kaji', 'teliti', 'kena', ...",pengaruh kerja layan prima center of informati...,masalah jadi kaji teliti kena layan prima inti...
4,2541-3279-5,kontribusi literasi informasi mahasiswa terhad...,fenomena pesatnya perkembangan ilmu pengetahua...,kontribusi literasi informasi mahasiswa terhad...,fenomena pesatnya perkembangan ilmu pengetahua...,"['kontribusi', 'literasi', 'informasi', 'mahas...","['fenomena', 'pesat', 'kembang', 'ilmu', 'tahu...",kontribusi literasi informasi mahasiswa proses...,fenomena pesat kembang ilmu tahu teknologi inf...


# Audit Dataset

In [3]:
# Check Duplicate Documents
df.duplicated().sum()

np.int64(0)

# Merging Title and Abstract into a One-Text

In [4]:
df["processed_text"] = df["processed_title"] + " " + df["processed_abstract"]

df[["processed_text"]].sample(5, random_state=42)

,processed_text
4680,analisis kontribusi pustaka indonesian visual ...
1033,tingkat puas mustaka mutu layan pustaka bas si...
1044,aktif pustakawan masyarakat perpusdokinfo ting...
3425,dampak rendah pustaka masa pandemi covid 19 si...
4319,representasi profesionalisme pustakawan film t...


In [5]:
vectorizer = TfidfVectorizer()

X_main = vectorizer.fit_transform(df["processed_text"])

print(f"Shape matriks TF-IDF (dokumen, term): {X_main.shape}")
print(f"Vocabulary size: {len(vectorizer.vocabulary_):,}")
print(f"Jumlah dokumen: {X_main.shape[0]:,}")
print(f"Jumlah term unik: {X_main.shape[1]:,}")


Shape matriks TF-IDF (dokumen, term): (5890, 23287)
Vocabulary size: 23,287
Jumlah dokumen: 5,890
Jumlah term unik: 23,287


In [6]:
n_rows, n_cols = X_main.shape
total_cells = n_rows * n_cols
sparsity_percent = 100 * (1 - (X_main.nnz / total_cells)) if total_cells else 0

matrix_summary = pd.DataFrame({
    "metric": [
        "documents (rows)",
        "terms (columns)",
        "non-zero TF-IDF values",
        "total possible cells",
        "sparsity (%)",
    ],
    "value": [n_rows, n_cols, X_main.nnz, total_cells, round(sparsity_percent, 4)],
})

display(matrix_summary)

,metric,value
0,documents (rows),5.890000e+03
1,terms (columns),2.328700e+04
2,non-zero TF-IDF values,4.497360e+05
3,total possible cells,1.371604e+08
4,sparsity (%),9.967210e+01


In [11]:

# Ambil nama fitur sesuai urutan kolom pada matriks TF-IDF
feature_names = vectorizer.get_feature_names_out()

vocabulary_output = pd.DataFrame({
    "feature_index": np.arange(len(feature_names), dtype=int),
    "term": feature_names,
    "idf": vectorizer.idf_,
})

print("20 term dengan IDF tertinggi:")
display(
    vocabulary_output
    .nlargest(20, "idf")
    .sort_values("idf", ascending=False)
)

print("20 term dengan IDF terendah:")
display(
    vocabulary_output
    .nsmallest(20, "idf")
    .sort_values("idf", ascending=True)
)


20 term dengan IDF tertinggi:


,feature_index,term,idf
2,2,0000,8.988034
3,3,0001,8.988034
4,4,0002,8.988034
5,5,0003,8.988034
6,6,000x,8.988034
9,9,003,8.988034
11,11,0046,8.988034
13,13,0050,8.988034
14,14,006,8.988034
20,20,012,8.988034


20 term dengan IDF terendah:


,feature_index,term,idf
7883,7883,guna,1.195479
21104,21104,teliti,1.231197
8089,8089,hasil,1.290459
17434,17434,pustaka,1.324391
21884,21884,tuju,1.374215
13704,13704,metode,1.499461
11564,11564,laku,1.549944
5106,5106,data,1.557030
8874,8874,informasi,1.591392
20733,20733,tahu,1.661897


# Export TF-IDF Results

In [10]:
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

save_npz(MATRIX_PATH, X_main, compressed=True)
joblib.dump(vectorizer, VECTORIZER_PATH)
df.to_csv(DOCUMENTS_PATH, index=False, encoding="utf-8-sig")

vocabulary_output.to_csv(VOCABULARY_PATH, index=False, encoding="utf-8-sig")

for path in [MATRIX_PATH, VECTORIZER_PATH, DOCUMENTS_PATH, VOCABULARY_PATH]:
    print(f"Saved: {path.resolve()} ({path.stat().st_size / (1024 * 1024):.2f} MB)")

Saved: C:\Works\uts-ir\src\data\features\tfidf\tfidf_matrix.npz (3.79 MB)
Saved: C:\Works\uts-ir\src\data\features\tfidf\tfidf_vectorizer.joblib (0.49 MB)
Saved: C:\Works\uts-ir\src\data\features\tfidf\tfidf_documents.csv (34.70 MB)
Saved: C:\Works\uts-ir\src\data\features\tfidf\tfidf_vocabulary.csv (0.75 MB)
